In [ ]:
from cmapss.config import load_config, raw_files
from cmapss.data import load_data, load_rul_labels, SENSOR_COLS, FD001_DEAD_SENSORS
from cmapss.features import (
    add_rul_column, split_units, fit_scaler, apply_scaler,
    create_sequences, create_test_sequences,
)

cfg = load_config()
files = raw_files(cfg)
pp = cfg["preprocess"]

RUL_CAP = pp["rul_cap"]          # 커뮤니티 표준값 (120~130 범위), 최소 수명 128과도 스케일이 맞음
WINDOW_SIZE = pp["window_size"]
VAL_RATIO = pp["val_ratio"]
SEED = pp["seed"]

In [ ]:
df = load_data(files["train"])
feature_cols = [i for i in SENSOR_COLS if i not in FD001_DEAD_SENSORS]
print(f"N of available features {len(feature_cols)}:", feature_cols)

# cap에 걸린 row 비율 극단적으로 높거나 낮으면 cap 재조정 신호
df = add_rul_column(df, rul_cap=RUL_CAP)
cap_ratio = (df["RUL"] == RUL_CAP).mean()
df[["unit", "cycle", "RUL"]].head()

In [ ]:
# Train/Val Split — unit(엔진) 단위
tr_df, val_df, tr_units, val_units = split_units(df, val_ratio=VAL_RATIO, seed=SEED)
overlap = set(tr_units) & set(val_units)

# Normalize
scaler = fit_scaler(tr_df, feature_cols)
tr_scaled = apply_scaler(tr_df, scaler, feature_cols)
val_scaled = apply_scaler(val_df, scaler, feature_cols)

# 슬라이딩 윈도우 생성 (train/val — 엔진당 여러 윈도우)
X_train, y_train, _ = create_sequences(tr_scaled, feature_cols, WINDOW_SIZE)
X_val, y_val, _ = create_sequences(val_scaled, feature_cols, WINDOW_SIZE)
print("X_train", X_train.shape, "X_val", X_val.shape)

In [ ]:
test_df = load_data(files["test"])
test_scaled = apply_scaler(test_df, scaler, feature_cols)
X_test, test_units = create_test_sequences(test_scaled, feature_cols, WINDOW_SIZE)
y_test = load_rul_labels(files["rul"])

print("X_test", X_test.shape, "N of the answers", len(y_test))
assert X_test.shape[0] == len(y_test)
print("Validation Successful: equalivant number of engines and answers")
assert X_val.shape[0] == len(y_val)

In [ ]:
# Save dataset
#`data/interim/`: RUL 라벨까지 붙인 중간 df (아직 배열은 아님)
#`data/processed/`: 모델에 바로 넣을 수 있는 최종 배열

import numpy as np

interim_dir = cfg["paths"]["interim_dir"]
processed_dir = cfg["paths"]["processed_dir"]
interim_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

tr_df.to_parquet(interim_dir / "train_with_rul.parquet")
val_df.to_parquet(interim_dir / "val_with_rul.parquet")

for name, arr in {"X_train": X_train, "y_train": y_train, "X_val": X_val,
                  "y_val": y_val, "X_test": X_test, "y_test": y_test}.items():
    np.save(processed_dir / f"{name}.npy", arr)
print("Saved at", interim_dir, "and", processed_dir)